In [2]:
import glob
import pandas as pd

csv_files = glob.glob('*.csv')

for filename in csv_files:
    print(f"📄 文件名: {filename}")
    try:
        # 只读取前 2 行 (nrows=1 读取数据行，header默认是第一行)
        # 注意：pd.read_csv 默认把第一行当做表头，nrows=1 会读入表头 + 1行数据
        df = pd.read_csv(filename, nrows=1)
        
        # 打印表头（列名）
        print(f"   第一行 (列名): {list(df.columns)}")
        
        # 打印第一行数据 (如果文件不为空)
        if not df.empty:
            print(f"   第二行 (数值): {df.iloc[0].tolist()}")
            
    except Exception as e:
        print(f"   读取出错: {e}")
    print("-" * 40)

📄 文件名: kg.csv
   第一行 (列名): ['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source', 'y_index', 'y_id', 'y_type', 'y_name', 'y_source']
   第二行 (数值): ['protein_protein', 'ppi', 0, 9796, 'gene/protein', 'PHYHIP', 'NCBI', 8889, 56992, 'gene/protein', 'KIF15', 'NCBI']
----------------------------------------
📄 文件名: NACC_ad.csv
   第一行 (列名): ['path', 'filename', 'ID', 'age', 'gender', 'education', 'hispanic', 'race', 'apoe', 'NC', 'MCI', 'DE', 'COG', 'AD', 'mmse', 'moca', 'cdr', 'cdrSum', 'boston', 'digitB', 'digitBL', 'digitF', 'digitFL', 'animal', 'Fwords', 'gds', 'lm_imm', 'lm_del', 'lm_memtime', 'craft_imm', 'craft_del', 'mint', 'numberB', 'numberBL', 'numberF', 'numberFL', 'trailA', 'trailB', 'npiq_DEL', 'npiq_HALL', 'npiq_AGIT', 'npiq_DEPD', 'npiq_ANX', 'npiq_ELAT', 'npiq_APA', 'npiq_DISN', 'npiq_IRR', 'npiq_MOT', 'npiq_NITE', 'npiq_APP', 'faq_BILLS', 'faq_TAXES', 'faq_SHOPPING', 'faq_GAMES', 'faq_STOVE', 'faq_MEALPREP', 'faq_EVENTS', 'faq_PAYATTN', 'faq_REMDA

In [3]:
import pandas as pd

# ================= 配置 =================
INPUT_KG = 'kg.csv'  # 你的原始大文件
# 我们需要查找的关键词：包括阿尔茨海默、痴呆、认知障碍、APOE基因
KEYWORDS = ['Alzheimer', 'Dementia', 'Cognitive', 'APOE']
# =======================================

def search_keywords():
    print(f"🔍 正在读取 {INPUT_KG} 并搜索关键词: {KEYWORDS} ...")
    
    found_nodes = set()
    chunksize = 100000
    
    # 为了避免输出太多，我们只打印前几条，但会收集所有不重复的匹配项
    for chunk in pd.read_csv(INPUT_KG, chunksize=chunksize):
        # 转换成字符串处理，防止报错
        x_str = chunk['x_name'].astype(str)
        y_str = chunk['y_name'].astype(str)
        
        # 构建掩码：只要 x_name 或 y_name 包含任一关键词
        mask = pd.Series(False, index=chunk.index)
        for kw in KEYWORDS:
            mask |= x_str.str.contains(kw, case=False, na=False)
            mask |= y_str.str.contains(kw, case=False, na=False)
        
        if mask.any():
            subset = chunk[mask]
            # 打印一些示例看看长什么样
            if len(found_nodes) < 5: 
                print("\n✅ 找到相关示例 (Top 5):")
                print(subset[['x_name', 'relation', 'y_name']].head(5))
            
            # 收集找到的所有相关节点名字，方便我们人工挑选
            # 这里的逻辑是：把这行里包含关键词的那个名字挑出来
            for kw in KEYWORDS:
                # 找 x_name 里的匹配
                matches_x = subset.loc[subset['x_name'].str.contains(kw, case=False, na=False), 'x_name']
                found_nodes.update(matches_x.tolist())
                
                # 找 y_name 里的匹配
                matches_y = subset.loc[subset['y_name'].str.contains(kw, case=False, na=False), 'y_name']
                found_nodes.update(matches_y.tolist())
    
    if not found_nodes:
        print("❌ 未找到任何相关节点。")
    else:
        print("\n📋 候选节点列表 (请从中挑选准确的名称用于下一步):")
        # 排序并打印，方便查看
        sorted_nodes = sorted(list(found_nodes))
        # 只打印前30个和后10个，或者打印包含特定词的核心词
        print("--- Alzheimer 相关 ---")
        print([n for n in sorted_nodes if 'alzheimer' in n.lower()][:10])
        print("--- Dementia 相关 ---")
        print([n for n in sorted_nodes if 'dementia' in n.lower()][:10])
        print("--- Cognitive 相关 ---")
        print([n for n in sorted_nodes if 'cognitive' in n.lower()][:10])
        print("--- APOE 相关 ---")
        print([n for n in sorted_nodes if 'apoe' in n.lower()][:10])

if __name__ == "__main__":
    search_keywords()

🔍 正在读取 kg.csv 并搜索关键词: ['Alzheimer', 'Dementia', 'Cognitive', 'APOE'] ...

✅ 找到相关示例 (Top 5):
      x_name         relation  y_name
7544    APOE  protein_protein  ZNF451
8590    APOE  protein_protein   PSEN1
10307   APOE  protein_protein   PCMT1
19723   APOE  protein_protein   HIPK1
20366   APOE  protein_protein   LOXL4

✅ 找到相关示例 (Top 5):
       x_name         relation y_name
101793   APOE  protein_protein  VDAC1
104349   APOE  protein_protein  FOXG1
109020   APOE  protein_protein   RFX5
110370   APOE  protein_protein     HP
110438   APOE  protein_protein   EPN2

✅ 找到相关示例 (Top 5):
       x_name         relation y_name
200819   APOE  protein_protein  FXYD7
203739   APOE  protein_protein  VCAM1
209567    ALB  protein_protein   APOE
218915   APOE  protein_protein  PIAS1
227932   APOE  protein_protein   GCDH

✅ 找到相关示例 (Top 5):
       x_name         relation   y_name
306066   APOE  protein_protein  MID1IP1
307847   APOE  protein_protein    VDAC3
309320   APOE  protein_protein     RHEB
311717 

C:\Users\86137\AppData\Local\Temp\ipykernel_7544\2271651797.py:16: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(INPUT_KG, chunksize=chunksize):
C:\Users\86137\AppData\Local\Temp\ipykernel_7544\2271651797.py:16: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(INPUT_KG, chunksize=chunksize):
C:\Users\86137\AppData\Local\Temp\ipykernel_7544\2271651797.py:16: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(INPUT_KG, chunksize=chunksize):
C:\Users\86137\AppData\Local\Temp\ipykernel_7544\2271651797.py:16: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(INPUT_KG, chunksize=chunksize):
C:\Users\86137\AppData\Local\Temp\ipykernel_7544\2271651797.py:16: DtypeWarning: Columns (3,8) h


📋 候选节点列表 (请从中挑选准确的名称用于下一步):
--- Alzheimer 相关 ---
['Alzheimer disease', 'Alzheimer disease without neurofibrillary tangles', 'Alzheimer disease, familial early-onset, with coexisting amyloid and prion pathology', 'Alzheimer disease, susceptibility to, mitochondrial', "Deregulated CDK5 triggers multiple neurodegenerative pathways in Alzheimer's disease models", 'dementia/parkinsonism with non-Alzheimer amyloid plaques', 'familial Alzheimer disease']
--- Dementia 相关 ---
['AIDS dementia complex', 'Dementia', 'Frontal lobe dementia', 'Frontolimbic dementia', 'Frontotemporal dementia', 'Lewy body dementia', 'PRKAR1B-related neurodegenerative dementia with intermediate filaments', 'Semantic dementia', 'Subcortical dementia', 'amyotrophic lateral sclerosis-parkinsonism-dementia complex']
--- Cognitive 相关 ---
['Autism with high cognitive abilities', 'Cognitive epileptic aura', 'Cognitive fatigue', 'Cognitive impairment', 'Focal aware cognitive seizure', 'Focal aware cognitive seizure with anom